# Prática 17/11 · Qual modelo escolher? Validação cruzada, árvores e sobreajuste
**Aprendizagem Computacional · 4º TIA · Aula 04 · versão INICIAL**

Em 10/11, a regressão linear venceu as duas réguas numa validação de **28 municípios**. Mas uma validação só depende de quais municípios caíram nela. Hoje:

1. vemos o problema: mudar o sorteio muda o resultado;
2. trocamos a validação única por **validação cruzada** (5 dobras, só no treino);
3. vemos o **sobreajuste** na prática, com árvores de profundidade crescente;
4. comparamos 7 candidatos no mesmo protocolo e ajustamos o k do k-NN com `GridSearchCV`;
5. escolhemos o modelo e só então abrimos o teste, **uma única vez**.

É o que o grupo mostra no check-in de 24/11: 2 ou 3 modelos comparados com validação cruzada, e o teste usado uma vez.

| TODO | Tarefa | Apostila |
|---|---|---|
| 1 | `KFold` com 5 dobras | cap. 5 |
| 2 | Curva de complexidade da árvore (`cross_validate` com `return_train_score`) | cap. 10 e 13 |
| 3 | Montar k-NN (com escala) e Random Forest | cap. 11 e 16 |
| 4 | `GridSearchCV` para o k do k-NN | cap. 11 |
| 5 | Ajustar o escolhido em todo o treino e prever o teste uma vez | cap. 5 |

## 1 · Carregar a base e repetir o split de 10/11 (pronto)
Mesma base, mesmo X e y, mesmo teste guardado.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
SEMENTE = 42
pd.set_option('display.precision', 2)

ARQUIVO = 'base_analitica.csv'
URL = 'https://raw.githubusercontent.com/Schavess/ifmt-vg-ac/main/atividades/2026-11-17_Aula04_Validacao_Cruzada_Arvores/dados/' + ARQUIVO
candidatos = [Path('dados') / ARQUIVO, Path('..') / 'dados' / ARQUIVO, Path(ARQUIVO)]
local = next((p for p in candidatos if p.exists()), None)
base = pd.read_csv(local if local is not None else URL, dtype={'CD_MUNICIPIO': str})
print('Base lida de:', local.resolve() if local is not None else URL, '|', base.shape)

In [ ]:
Y_COL = 'ABSTENCAO_PCT'
X_COLS = ['PCT_16_24', 'PCT_65_MAIS', 'PCT_SUPERIOR_COMPLETO', 'LOG_ELEITORADO',
          'ABST_2022_PCT', 'CRESC_ELEITORADO_PCT', 'DIST_CUIABA_KM']
X, y = base[X_COLS], base[Y_COL]
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.20, random_state=SEMENTE)
print('Treino:', len(X_treino), '| Teste (guardado):', len(X_teste))

## 2 · O problema: o resultado depende do sorteio (pronto)
Vamos repetir o experimento de 10/11 (ajuste e validação dentro do treino) mudando **só** a semente do sorteio.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

maes = []
for semente in range(10):
    Xa, Xv, ya, yv = train_test_split(X_treino, y_treino, test_size=0.25, random_state=semente)
    modelo = Pipeline([('escala', StandardScaler()), ('modelo', LinearRegression())]).fit(Xa, ya)
    maes.append(mean_absolute_error(yv, modelo.predict(Xv)))
    print(f'semente {semente}: MAE na validação = {maes[-1]:.2f}')
print(f'menor {min(maes):.2f} | maior {max(maes):.2f}')

O mesmo modelo, com os mesmos dados, muda de erro só por causa do sorteio. Se escolhermos o modelo por **uma** validação, estaremos escolhendo, em parte, pela sorte.

**Validação cruzada (cap. 5):** dividimos o **treino** em 5 partes (dobras). Cada dobra serve uma vez de validação, enquanto as outras 4 ajustam o modelo. No fim, temos 5 erros: a **média** diz quanto o modelo erra, e o **desvio** diz o quanto isso depende do sorteio.

## 3 · TODO 1: as 5 dobras (cap. 5)
Crie `cv = KFold(n_splits=5, shuffle=True, random_state=SEMENTE)`. O `shuffle=True` embaralha antes de cortar, porque a base está ordenada por código do município.

In [ ]:
from sklearn.model_selection import KFold, cross_validate

cv = None  # TODO 1: KFold com 5 dobras, shuffle=True, random_state=SEMENTE
for i, (idx_aj, idx_val) in enumerate(cv.split(X_treino), start=1):
    print(f'dobra {i}: ajusta em {len(idx_aj)}, valida em {len(idx_val)}')

## 4 · TODO 2: sobreajuste na prática, com árvores (cap. 10 e 13)
Uma árvore de decisão faz perguntas do tipo "abstenção de 2022 > 27%?" e divide os municípios até chegar a grupos pequenos. A **profundidade** é quantas perguntas seguidas ela pode fazer.

Árvore rasa demais não aprende o padrão (**subajuste**). Árvore funda demais decora os municípios do ajuste, inclusive o ruído (**sobreajuste**): o erro no ajuste vai a zero, e o erro na validação sobe.

Dentro do `for`, chame `cross_validate(arvore, X_treino, y_treino, cv=cv, scoring='neg_mean_absolute_error', return_train_score=True)`. O scikit-learn devolve o erro **negativo** (para "maior ser melhor"); por isso o sinal de menos ao guardar.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

profundidades = [1, 2, 3, 4, 5, 6, 8, None]
curva = []
for p in profundidades:
    arvore = DecisionTreeRegressor(max_depth=p, random_state=SEMENTE)
    r = None  # TODO 2: cross_validate(...) com scoring='neg_mean_absolute_error' e return_train_score=True
    curva.append({'profundidade': str(p), 'MAE_ajuste': -r['train_score'].mean() + 0.0, 'MAE_validacao': -r['test_score'].mean()})
curva = pd.DataFrame(curva)
ax = curva.plot(x='profundidade', y=['MAE_ajuste', 'MAE_validacao'], marker='o', figsize=(7, 3.5))
ax.set_ylabel('MAE (p.p.)'); ax.set_title('Árvore: erro no ajuste × na validação'); plt.tight_layout(); plt.show()
curva

**Pergunta para discussão:** a árvore sem limite (`None`) tem erro **zero** no ajuste. Por que ela não é o melhor modelo?

**Resposta esperada:** porque ela decorou os 90 municípios de cada ajuste, um por folha. Erro zero no ajuste não diz nada sobre municípios novos; na validação, ela erra mais do que a árvore de profundidade 4. É como uma equação de volume que passa exatamente por todas as árvores cubadas, mas erra feio numa árvore nova do talhão.

## 5 · TODO 3: os candidatos (cap. 11 e 16)
As réguas, a linear e a árvore (com a profundidade de menor erro na validação) já estão no dicionário. Complete:
- **k-NN:** `Pipeline([('escala', StandardScaler()), ('modelo', KNeighborsRegressor(n_neighbors=5))])`. O k-NN mede distância entre municípios, então **precisa** de escala (cap. 7).
- **Random Forest:** `RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=SEMENTE)`. São 300 árvores, cada uma treinada numa amostra diferente; a previsão é a média delas.

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

class Persistencia(BaseEstimator, RegressorMixin):
    """Baseline "a abstenção se repete": abstenção de 2022 + a variação média vista no treino."""
    def __init__(self, coluna='ABST_2022_PCT'):
        self.coluna = coluna
    def fit(self, X, y):
        self.desloc_ = float(np.mean(np.asarray(y) - X[self.coluna].to_numpy()))
        return self
    def predict(self, X):
        return X[self.coluna].to_numpy() + self.desloc_

melhor_prof = curva.loc[curva['MAE_validacao'].idxmin(), 'profundidade']
candidatos = {
    'Baseline: média': DummyRegressor(strategy='mean'),
    'Baseline: persistência': Persistencia(),
    'Regressão linear': Pipeline([('escala', StandardScaler()), ('modelo', LinearRegression())]),
    'Ridge (alpha=1)': Pipeline([('escala', StandardScaler()), ('modelo', Ridge(alpha=1.0))]),
    f'Árvore (prof. {melhor_prof})': DecisionTreeRegressor(max_depth=int(melhor_prof), random_state=SEMENTE),
    'k-NN (k=5)': None,      # TODO 3: Pipeline com StandardScaler e KNeighborsRegressor(n_neighbors=5)
    'Random Forest': None,   # TODO 3: RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=SEMENTE)
}
print(list(candidatos))

## 6 · Comparação justa (pronto)
Todos os candidatos passam pelo **mesmo** protocolo: mesmas dobras, mesma métrica, só o treino.

In [ ]:
linhas = []
for nome, modelo in candidatos.items():
    r = cross_validate(modelo, X_treino, y_treino, cv=cv, scoring='neg_mean_absolute_error')
    linhas.append({'modelo': nome, 'MAE_cv': -r['test_score'].mean(), 'desvio': r['test_score'].std()})
tabela_cv = pd.DataFrame(linhas).set_index('modelo').sort_values('MAE_cv')
tabela_cv

**Pergunta para discussão:** a Random Forest, que costuma ganhar competições, perde para a regressão linear aqui. Por quê?

**Resposta esperada:** com só 112 municípios e uma relação quase em linha reta entre a abstenção de 2022 e a de 2026, a linear já captura o principal. A floresta divide os dados em caixas e precisa de muitos exemplos para desenhar uma reta com degraus; com poucos dados, os degraus ficam grosseiros. Modelo flexível não é sinônimo de modelo melhor: depende do tamanho e da forma dos dados.

## 7 · TODO 4: o melhor k do k-NN com `GridSearchCV` (cap. 11)
O k (quantos vizinhos entram na média) é um **hiperparâmetro**: nós escolhemos, o modelo não aprende. O `GridSearchCV` testa cada valor da lista com a mesma validação cruzada e guarda o melhor.

Complete: `GridSearchCV(pipeline_knn, grade, cv=cv, scoring='neg_mean_absolute_error').fit(X_treino, y_treino)`.

In [ ]:
from sklearn.model_selection import GridSearchCV

pipeline_knn = Pipeline([('escala', StandardScaler()), ('modelo', KNeighborsRegressor())])
grade = {'modelo__n_neighbors': [1, 3, 5, 10, 15, 20, 30]}
busca = None  # TODO 4: GridSearchCV(...) ajustado em X_treino, y_treino
print('melhor k:', busca.best_params_['modelo__n_neighbors'], '| MAE na CV:', round(-busca.best_score_, 2))

## 8 · TODO 5: escolher e abrir o teste UMA vez (cap. 5)
Pela tabela, o candidato com menor erro na validação cruzada é a **regressão linear** (a Ridge empata, a diferença é menor que o desvio). Ficamos com a linear, que é mais simples de explicar. Só agora, com a escolha feita, olhamos o teste:
1. ajuste `candidatos['Regressão linear']` em **todo** o treino (`fit(X_treino, y_treino)`);
2. preveja `X_teste` e calcule o MAE.

As réguas também são medidas no teste, para a tabela final.

In [ ]:
from sklearn.metrics import r2_score

modelo_final = None   # TODO 5: candidatos['Regressão linear'] ajustado em TODO o treino
prev_final = None     # TODO 5: previsões do modelo_final em X_teste
teste = {'Regressão linear': prev_final,
         'Baseline: persistência': candidatos['Baseline: persistência'].fit(X_treino, y_treino).predict(X_teste),
         'Baseline: média': candidatos['Baseline: média'].fit(X_treino, y_treino).predict(X_teste)}
tabela_teste = pd.DataFrame({n: {'MAE_teste': mean_absolute_error(y_teste, p), 'R2_teste': r2_score(y_teste, p)}
                             for n, p in teste.items()}).T
tabela_teste

**Pergunta para discussão:** por que não escolhemos o modelo pelo resultado no teste, já que ele é "o mais real"?

**Resposta esperada:** porque, se o teste ajuda a escolher, ele deixa de ser independente: escolhemos o que deu sorte naqueles 29 municípios, e o número reportado fica otimista. O teste só mede o modelo que **já foi escolhido**. É o mesmo motivo de não divulgar o gabarito antes da prova.

## Verificação final
Rode sem alterar. Os valores esperados estão no `LEIAME.md` da prática.

In [ ]:
def conferir(descricao, teste):
    try:
        passou = bool(teste())
    except Exception:          # variável ainda vazia (None ou ...) ou célula não rodada
        passou = False
    print('✅' if passou else '❌', descricao)
    return passou

ok = [
    conferir('cv tem 5 dobras embaralhadas', lambda: cv.get_n_splits() == 5 and cv.shuffle),
    conferir('curva da árvore: sem limite, erro zero no ajuste', lambda: round(curva.loc[curva['profundidade'] == 'None', 'MAE_ajuste'].item(), 2) == 0),
    conferir('curva da árvore: melhor profundidade = 4', lambda: str(melhor_prof) == '4'),
    conferir('7 candidatos montados (nenhum None)', lambda: len(candidatos) == 7 and all(m is not None for m in candidatos.values())),
    conferir('regressão linear: MAE 1,33 na CV', lambda: round(tabela_cv.loc['Regressão linear', 'MAE_cv'], 2) == 1.33),
    conferir('Random Forest: MAE 1,62 na CV', lambda: round(tabela_cv.loc['Random Forest', 'MAE_cv'], 2) == 1.62),
    conferir('GridSearch escolheu k = 5', lambda: busca.best_params_['modelo__n_neighbors'] == 5),
    conferir('teste: MAE da linear = 1,23', lambda: round(mean_absolute_error(y_teste, prev_final), 2) == 1.23),
]
print(f'\n{sum(ok)} de {len(ok)} verificações OK')

## Checklist (as perguntas da banca, para este experimento)
1. **y?** abstenção de 2026 (%). 2. **Baselines?** média e persistência. 3. **Métrica?** MAE em p.p., com média e desvio entre dobras.
4. **Split?** teste separado no início; escolha feita só com validação cruzada no treino; teste aberto uma vez. 5. **Vazamento?** escala dentro do `Pipeline` em cada dobra; contagens de 2026 fora de X.
6. **Evidência?** tabela da CV (escolha) e tabela do teste (número final).
**O que falta (24/11):** prever **todos** os municípios sem otimismo e transformar a previsão em decisão.